# Experiment 65 - Combined Per-Class Oracle

**Building on:** exp63 (OOF cmAP=0.96286) and exp64 raw-variant negative control.

**Hypothesis:** exp63's low-prior candidates are strong globally, while exp64 showed some classes prefer MLP-only/raw-style candidates. Combine the exp63 candidate family with a small component-only/raw family and select per class by OOF padded AP.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp60_path = nb_dir / 'exp60_taxon_dual_power.ipynb'
exp60_nb = json.loads(exp60_path.read_text())

for cell_idx in [1, 2]:
    print(f'--- executing exp60 cell {cell_idx} ---')
    src = ''.join(exp60_nb['cells'][cell_idx]['source'])
    if cell_idx == 2:
        src = src.split("best_cmap = -1.0")[0]
    exec(compile(src, f'exp60_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp65: combined oracle over exp63 low-prior family plus raw/component variants

def padded_ap_per_class(y_true, y_pred, pad=5):
    aps = np.zeros(y_true.shape[1], dtype=np.float32)
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps[c] = average_precision_score(yt, yp)
    return aps

best_ap = np.full(N_CLASSES, -1.0, dtype=np.float32)
best_pred = np.zeros_like(oof_raw_0, dtype=np.float32)
best_meta = [None] * N_CLASSES
global_rows = []
prior_cache = {}

def update_candidate(probs, cfg):
    global best_ap, best_pred, best_meta
    aps = padded_ap_per_class(Y_FULL_aligned, probs)
    cmap = float(aps.mean())
    global_rows.append({**cfg, 'cmap': cmap})
    improved = aps > best_ap
    if improved.any():
        best_ap[improved] = aps[improved]
        best_pred[:, improved] = probs[:, improved]
        for ci in np.where(improved)[0]:
            best_meta[ci] = cfg.copy()

def get_prior_pair(lam_event, lam_texture):
    key = (lam_event, lam_texture)
    if key not in prior_cache:
        prior_cache[key] = (
            apply_prior_taxon(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables,
                              lambda_event=lam_event, lambda_texture=lam_texture),
            apply_prior_taxon(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables,
                              lambda_event=lam_event, lambda_texture=lam_texture),
        )
    return prior_cache[key]

n_candidates = 0

# Strong low-prior family from exp63.
LAM_EVENTS = [0.4, 0.8, 1.2, 1.6, 2.2]
LAM_TEXTURES = [2.0, 4.0, 6.0, 8.0, 10.0]
TTA_W0S = [0.50, 0.75, 0.90, 1.00]
POWER_PAIRS = [(0.3, 0.8), (0.5, 1.0), (0.7, 1.2), (0.9, 1.2), (0.7, 1.4), (1.1, 1.2)]
ALPHA_TEXTURES = [0.20, 0.30]
ENSEMBLE_GRIDS = [(0.00, 0.65, 0.35), (0.02, 0.63, 0.35), (0.02, 0.58, 0.40), (0.02, 0.56, 0.42), (0.05, 0.60, 0.35)]

for lam_event in LAM_EVENTS:
    for lam_texture in LAM_TEXTURES:
        prior_0_c, prior_250_c = get_prior_pair(lam_event, lam_texture)
        for wp, wm, wpr in ENSEMBLE_GRIDS:
            fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0_c
            fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250_c
            for tta_w0 in TTA_W0S:
                fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
                for power_event, power_texture in POWER_PAIRS:
                    for alpha_texture in ALPHA_TEXTURES:
                        probs = postprocess_taxon_dual_power(fp_avg, power_event=power_event,
                            power_texture=power_texture, alpha_event=0.10, alpha_texture=alpha_texture)
                        update_candidate(probs, dict(family='low_prior', lam_event=lam_event, lam_texture=lam_texture,
                            tta_w0=tta_w0, power_event=power_event, power_texture=power_texture,
                            alpha_texture=alpha_texture, wp=wp, wm=wm, wpr=wpr))
                        n_candidates += 1

# Raw/component-only family from exp64, but small.
for tta_w0 in [0.50, 0.90, 1.00]:
    components = {
        'raw': tta_w0 * oof_raw_0 + (1.0 - tta_w0) * oof_raw_250,
        'mlp': tta_w0 * oof_mlp_0 + (1.0 - tta_w0) * oof_mlp_250,
        'proto': tta_w0 * oof_proto_0 + (1.0 - tta_w0) * oof_proto_250,
    }
    for name, logits in components.items():
        for power_event, power_texture in [(0.0, 0.0), (0.3, 0.8), (0.7, 1.2)]:
            for alpha_texture in [0.0, 0.2]:
                probs = postprocess_taxon_dual_power(logits, power_event=power_event,
                    power_texture=power_texture, alpha_event=0.10, alpha_texture=alpha_texture)
                update_candidate(probs, dict(family=name, lam_event=np.nan, lam_texture=np.nan,
                    tta_w0=tta_w0, power_event=power_event, power_texture=power_texture,
                    alpha_texture=alpha_texture, wp=np.nan, wm=np.nan, wpr=np.nan))
                n_candidates += 1

oracle_cmap = padded_cmap(Y_FULL_aligned, best_pred)
oracle_auc = macro_auc(Y_FULL_aligned, best_pred)
global_df = pd.DataFrame(global_rows).sort_values('cmap', ascending=False).reset_index(drop=True)
selected = pd.DataFrame(best_meta)

print(f'Evaluated candidates: {n_candidates}')
print(f'Best single-candidate cmAP: {global_df.iloc[0].cmap:.5f}')
print(global_df.head(15).to_string(index=False))
print('=' * 60)
print('Experiment 65 - Combined per-class oracle')
print(f'AUC={oracle_auc:.5f}  cmAP={oracle_cmap:.5f}')
print(f'vs exp63 OOF (0.96286): delta cmAP = {oracle_cmap - 0.96286:+.5f}')
print(f'vs target 0.97000: delta cmAP = {oracle_cmap - 0.97000:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
print('\nSelected family counts:')
print(selected['family'].value_counts().to_string())
print('\nSelected low-prior lambda counts:')
lp = selected[selected['family'] == 'low_prior']
print(lp.groupby(['lam_event', 'lam_texture']).size().sort_values(ascending=False).head(15).to_string())
